# Registries, Tags & Distribution

## What's covered

- **What a registry actually is** — a server speaking the OCI distribution API; not magic
- **Docker Hub and the alternatives** — GHCR, ECR, GAR, ACR, Quay, Harbor
- **The push/pull workflow** — `docker tag`, `docker push`, what the daemon uploads
- **Authentication** — `docker login`, `~/.docker/config.json`, credential helpers, registry tokens
- **Image references revisited** — registry, namespace, repo, tag, digest, and what each piece controls
- **Tagging conventions in practice** — `latest`, semver, sliding tags, git SHAs, channels
- **Image manifests and digests** — what actually gets pushed, why digests are immutable
- **Manifest lists (OCI image index)** — one tag, many architectures
- **Content addressability and dedup** — why pushing a 200 MB image is sometimes free
- **Running a local registry** — `registry:2` for air-gapped or just for testing
- **Mirroring and pull-through caching** — making your local network the "default" registry
- **Garbage collection on a registry** — how blobs get cleaned up
- **Image scanning preview** — vulnerability scans (deeper in notebook 08)

By the end you should be able to push and pull images against any registry, set up a private one in five minutes, and explain exactly what bytes leave your laptop when you `docker push`.

## What a registry actually is

A **container registry** is a web service that stores and serves images, speaking a specific HTTP API called the **OCI Distribution Specification** (formerly the "Docker Registry HTTP API v2"). Every registry — Docker Hub, GHCR, ECR, your private one — implements the same protocol. The `docker` CLI doesn't know which registry it's talking to; it just speaks the standard.

Two things a registry stores:

1. **Blobs.** Content-addressable byte storage. Layers are blobs. Image configs are blobs. Each blob is identified by its SHA-256 digest (`sha256:abc123...`). Uploading the same bytes twice is a no-op — the second upload sees the digest already exists and skips.
2. **Manifests.** Small JSON documents that describe an image: a list of layer digests, the config blob digest, the platform, and a few labels. The manifest itself has a digest. **A "tag" is just a mutable pointer from a name like `1.27.0` to a manifest digest.**

So an image push consists of:

```
1. Upload each layer's blob (deduped by digest -- only new layers cross the wire)
2. Upload the image config blob
3. Upload the manifest that lists them
4. Update the tag pointer to the manifest's digest
```

That's it. Pulls are the same steps reversed. There's no compression magic, no proprietary protocol — just HTTP GETs and PUTs against a digest-based content store.

Once you understand that, the rest of this notebook is mostly "which registry, which credentials, which tag conventions." The mechanics are always the same.

## Docker Hub and the alternatives

**Docker Hub** (`docker.io`) is the default. If you `docker pull nginx`, you're pulling from Hub. Free for public images; rate-limited for anonymous pulls (since 2020, around 100 pulls per IP per 6 hours). Paid tier raises the cap.

The major alternatives:

| Registry | Hostname | Best for |
|---|---|---|
| **GitHub Container Registry** | `ghcr.io` | Anything that lives on GitHub. Free for public, integrated with GitHub Actions and packages. The most common Hub alternative for OSS today. |
| **AWS Elastic Container Registry** | `<acct>.dkr.ecr.<region>.amazonaws.com` | AWS workloads. Tight IAM integration. |
| **Google Artifact Registry** | `<region>-docker.pkg.dev` | GCP workloads. Replaces the older `gcr.io`. |
| **Azure Container Registry** | `<name>.azurecr.io` | Azure workloads. |
| **Quay** (Red Hat) | `quay.io` | OpenShift, Red Hat ecosystem. |
| **Harbor** | self-hosted | On-prem private registry with RBAC, scanning, replication. The standard self-hosted option. |
| **Distribution / `registry:2`** | self-hosted | The plain OSS reference registry. Run for testing or for an air-gapped network. |

You can mix freely. A team might build in CI, push to GHCR for storage, mirror to ECR for prod-side pull speed, and have a local Harbor for on-prem clusters. Same images, same digests — only the URL changes.

## The push/pull workflow

The full cycle for getting an image you built onto a registry:

```bash
# 1. Tag your local image with the registry-qualified name
docker tag flask-demo:0.1 ghcr.io/myorg/flask-demo:0.1
docker tag flask-demo:0.1 ghcr.io/myorg/flask-demo:latest

# 2. Authenticate to the registry
docker login ghcr.io

# 3. Push
docker push ghcr.io/myorg/flask-demo:0.1
docker push ghcr.io/myorg/flask-demo:latest

# 4. (elsewhere) Pull and run
docker pull ghcr.io/myorg/flask-demo:0.1
docker run ghcr.io/myorg/flask-demo:0.1
```

### What gets uploaded

When you `docker push`, the daemon walks the image's layers in order. For each:

- **Compute the layer's digest.** It already has this from the build.
- **Ask the registry "do you have `sha256:abc...`?"** A HEAD request.
- **If yes — skip.** No bytes uploaded.
- **If no — upload.** Stream the layer to the registry.

So pushing a new build that shares 90% of its layers with the previous build uploads only the 10% that changed. The shared base layers were already there. This is **content-addressable dedup**, and it's why `docker push` can be much faster than the image size suggests.

### What `docker tag` does

`docker tag SOURCE TARGET` doesn't copy data. It adds another name pointing at the same image (same manifest digest). After the two `tag` calls above, both `flask-demo:0.1` and `ghcr.io/myorg/flask-demo:0.1` refer to the same bytes on disk — same `docker image inspect`, same digest.

This is why **tagging is free** and why teams tag the same image with multiple names (semver, git SHA, `latest`) before pushing.

## Authentication

`docker login REGISTRY` prompts for username and password (or a token), then stores credentials in `~/.docker/config.json`. Subsequent `docker pull`/`push` to that registry are authenticated automatically.

```bash
docker login                       # defaults to docker.io
docker login ghcr.io              # specific registry
docker login -u USER -p TOKEN ghcr.io   # non-interactive (avoid in shells with history)
echo $TOKEN | docker login -u USER --password-stdin ghcr.io   # safer, via stdin
docker logout ghcr.io             # forget credentials
```

### `~/.docker/config.json`

```json
{
  "auths": {
    "ghcr.io": {
      "auth": "base64-of-user:token"
    }
  },
  "credsStore": "osxkeychain"
}
```

Storing a base64 of `user:token` in a file is **not** secret storage. Anyone reading the file has your credentials.

### Credential helpers (the right answer)

A **credential helper** offloads storage to the OS keychain or a secret manager:

- **macOS** — `docker-credential-osxkeychain` (Docker Desktop installs it).
- **Windows** — `docker-credential-wincred`.
- **Linux** — `docker-credential-pass` (uses `gnupg`+`pass`) or `docker-credential-secretservice`.
- **Cloud-specific** — `docker-credential-ecr-login` (uses AWS IAM), `docker-credential-gcr` (GCP), `docker-credential-acr-env` (Azure).

The `credsStore` field in `config.json` names the helper. With one configured, `auths.ghcr.io.auth` disappears from the file — the helper holds it instead.

### Per-registry credentials and CI tokens

For CI, use **per-registry tokens** instead of passwords:

- **Docker Hub** — Personal Access Tokens.
- **GHCR** — `GITHUB_TOKEN` (auto-injected in GitHub Actions) or PATs with `read:packages`/`write:packages` scopes.
- **ECR** — short-lived tokens from `aws ecr get-login-password`, refreshed by the credential helper.

Hard-coded passwords in CI scripts are the most common Docker security failure. Use tokens; scope them tightly; rotate them on schedule.

## Image references revisited

From notebook 02, recall the reference grammar:

```
[registry-host[:port]/][namespace/]repository[:tag][@digest]
```

Now we can fill in the details for what each piece controls.

| Piece | Default | Determines |
|---|---|---|
| `registry-host:port` | `docker.io` | Which registry's API to hit. |
| `namespace` | `library` (on Docker Hub) | Which user/org the image belongs to. |
| `repository` | required | The image's name within the namespace. |
| `tag` | `latest` | Which manifest the registry points to *right now*. **Mutable.** |
| `@digest` | none | A specific manifest digest. **Immutable.** Wins over tag if both present. |

A few real-world examples and what they imply:

- **`nginx`** — Docker Hub, `library/nginx`, latest tag. Probably the freshest stable Nginx.
- **`ghcr.io/myorg/api:v1.2.3`** — GHCR, myorg, `api` repo, tag `v1.2.3`. Could be re-pushed at any time.
- **`ghcr.io/myorg/api@sha256:abc123...`** — same image, but pinned. Re-pushes of `v1.2.3` won't change what this resolves to.
- **`123456.dkr.ecr.us-east-1.amazonaws.com/billing/processor:prod-2026-06-04`** — private ECR, dated channel tag. Production deploys often look like this.

**Tag + digest together** is the common production pattern: tag for human readability, digest for the byte-exact guarantee. Kubernetes manifests committed to git often look like:

```yaml
image: ghcr.io/myorg/api:v1.2.3@sha256:abc123...
```

The cluster pulls by digest; the tag is purely there for engineers reading the manifest.

## Tagging conventions in practice

Notebook 02 introduced the patterns. In a real org, you'll see them combined in a versioning scheme like this:

**At build time**, CI tags the freshly-built image with:

- The **git SHA**: `myapp:a1b2c3d`
- A **semver-from-tag** if the commit is a tagged release: `myapp:1.4.2`
- The **branch channel**: `myapp:main`, `myapp:rc`, `myapp:edge`

CI pushes all of them — they all point at the same manifest digest, so the cost is one upload.

**At deploy time**, the deploy manifest references the image by:

- **Digest** (`myapp@sha256:...`) — for reproducibility
- Or **immutable semver** (`myapp:1.4.2`) — assuming your org enforces tag immutability

### The "is `:latest` ever OK?" question

For images **you publish** — almost never. `latest` is a moving target; downstream users get random versions.

For images **you consume locally** — fine. `docker run nginx` saves typing.

For **production deploys** — never. Even if you trust yourself not to re-push `:latest`, a `docker pull` six months from now won't necessarily get the image you tested with.

### Tag immutability

Some registries can be configured to **reject re-pushes of an existing tag** (ECR has "immutable tags"; Harbor has "immutable repositories"). Turn this on for your release tags. Mistakes won't silently rewrite history.

## Image manifests and digests

When you push, the registry stores **blobs** (layers, config) and **manifests** (JSON pointing at the blobs). The manifest is the source of truth for "what is this image."

A simplified single-platform image manifest:

```json
{
  "schemaVersion": 2,
  "mediaType": "application/vnd.oci.image.manifest.v1+json",
  "config": {
    "mediaType": "application/vnd.oci.image.config.v1+json",
    "digest": "sha256:0123...",
    "size": 7023
  },
  "layers": [
    { "mediaType": "application/vnd.oci.image.layer.v1.tar+gzip",
      "digest": "sha256:abcd...", "size": 31405122 },
    { "mediaType": "...", "digest": "sha256:efgh...", "size": 2451 },
    { "mediaType": "...", "digest": "sha256:ijkl...", "size": 851 }
  ]
}
```

That JSON, exactly as serialised, is the **canonical bytes** of the image. SHA-256 those bytes and you get the image's digest. Change *anything* — a layer order, a byte in the config — and the digest changes. That's why `digest = identity`: it can't be tampered with without becoming a different image.

### Inspecting an image's manifest

`docker buildx imagetools inspect REF` shows the manifest:

```bash
docker buildx imagetools inspect nginx:alpine
docker buildx imagetools inspect nginx:alpine --raw   # print the actual JSON
```

For older Docker versions or compatibility, the `skopeo` tool is the standard way to introspect registries without pulling:

```bash
skopeo inspect docker://nginx:alpine
skopeo inspect --raw docker://nginx:alpine | jq
```

## Manifest lists — one tag, many architectures

A single tag like `nginx:alpine` works on an Intel Mac, an Apple Silicon Mac, an x86 server, and an ARM Raspberry Pi. How? The tag actually points at a **manifest list** (officially: an **OCI Image Index**), which is a thin JSON object listing per-architecture manifests:

```
nginx:alpine  -->  manifest list
                     |
                     +--> linux/amd64  -->  manifest A  -->  layers...
                     +--> linux/arm64  -->  manifest B  -->  layers...
                     +--> linux/arm/v7 -->  manifest C  -->  layers...
```

When you `docker pull nginx:alpine` on an ARM Mac, the daemon fetches the manifest list, picks the entry matching its platform (`linux/arm64`), and pulls *that* manifest's layers. The other architectures' bytes are never downloaded.

### Building a manifest list yourself

```bash
docker buildx create --name multi --use --bootstrap
docker buildx build \
  --platform linux/amd64,linux/arm64 \
  -t ghcr.io/myorg/api:1.2.3 \
  --push \
  .
```

`--push` is mandatory: a manifest list can only live in a registry, not in a local image store. After that one command, `ghcr.io/myorg/api:1.2.3` works on both architectures.

### Inspecting a manifest list

```bash
docker buildx imagetools inspect nginx:alpine
# Name:      docker.io/library/nginx:alpine
# MediaType: application/vnd.oci.image.index.v1+json
# Digest:    sha256:abc...
#
# Manifests:
#   Name:      docker.io/library/nginx:alpine@sha256:def...
#   MediaType: application/vnd.oci.image.manifest.v1+json
#   Platform:  linux/amd64
#   ...
```

## Running a local registry

For testing, air-gapped networks, or as a building block for a private registry, the OSS **`registry:2`** image runs a fully-functional OCI distribution server in one container.

```bash
# Start a registry on port 5000
docker run -d -p 5000:5000 --name registry --restart=always registry:2

# Tag and push to it
docker tag flask-demo:0.1 localhost:5000/flask-demo:0.1
docker push localhost:5000/flask-demo:0.1

# Pull from it (works from anywhere that can reach the host)
docker pull localhost:5000/flask-demo:0.1
```

That's a working registry in two commands.

### Persistence

Without a volume, blobs vanish when the container dies. Add storage:

```bash
docker run -d -p 5000:5000 \
  --name registry \
  --restart=always \
  -v registry-data:/var/lib/registry \
  registry:2
```

### TLS and auth

`registry:2` by default speaks plain HTTP. For non-localhost use you should put it behind a TLS-terminating proxy (Nginx, Caddy) and configure auth (htpasswd, OIDC). The image's docs walk through the env vars (`REGISTRY_AUTH=htpasswd`, etc).

For production, most teams reach for **Harbor** instead — it wraps `registry:2` with a UI, RBAC, replication, and built-in image scanning. Same protocol; just a richer front.

## Mirroring and pull-through caching

A common pattern at scale: your team pulls `nginx:alpine` thousands of times a day across CI and prod. Hitting Docker Hub for each pull is slow (latency, rate limits) and fragile (Docker Hub outage = your deploys fail).

The fix: a **pull-through cache** — a local registry that, on a miss, fetches from the upstream and caches the result.

```bash
docker run -d -p 5000:5000 \
  --name cache \
  -e REGISTRY_PROXY_REMOTEURL=https://registry-1.docker.io \
  -v cache-data:/var/lib/registry \
  registry:2
```

Now your CI configures Docker to use `localhost:5000` as a registry mirror (`daemon.json` → `"registry-mirrors": ["http://my-cache:5000"]`). The first pull of `nginx:alpine` traverses the cache and stores blobs; subsequent pulls are served from cache at LAN speed.

GHCR, ECR, GAR all support similar pull-through configurations via their own proxy modes. For organisations doing thousands of pulls/day this saves significant money and latency.

### Mirror vs proxy

Two related concepts:

- **Mirror** — the cache holds the *same* image references as upstream (`nginx:alpine` resolves the same way). The daemon picks the cache first and falls back to the original.
- **Proxy** — same idea, just named differently in some registries' UIs.

The Docker daemon flag is `registry-mirrors`. ECR and GAR have explicit "pull-through cache" features that handle the same workflow with cloud-managed storage.

## Garbage collection on a registry

A registry that's been running for months has thousands of blobs — many for tags that were overwritten or images that nobody references anymore. **Garbage collection** sweeps these.

For `registry:2`:

```bash
# Stop pushes during GC (or run in read-only mode)
docker exec registry registry garbage-collect \
  --delete-untagged \
  /etc/distribution/config.yml
```

What this does:

1. Walk all manifests in storage.
2. Walk all blobs referenced by those manifests.
3. **Delete blobs that aren't referenced.**

With `--delete-untagged`, also deletes manifests with no tag (the "untagged image" case from `docker images` shows `<none>`).

For Harbor and the cloud registries, GC is a UI/policy concern — you don't run it by hand, you configure retention rules ("keep last 10 tags", "delete builds older than 30 days that aren't `prod-*`").

### Why you'd care

Storage grows surprisingly fast. A CI pipeline pushing one image per commit accumulates a multi-gigabyte registry inside a quarter. Without GC and a retention policy, your registry storage bill just keeps climbing.

## Image scanning preview

Modern registries (Hub, ECR, GHCR, Harbor, Quay) scan pushed images against CVE databases and flag known vulnerabilities — usually with severity buckets and links to the upstream advisory.

The scan results live alongside the image and can be fetched via API. Common patterns:

- **CI gate** — fail the build if a new image introduces a `HIGH` or `CRITICAL` CVE that wasn't already present in the base image. Tools: `trivy`, `grype`, `snyk`, the registry's own scanner.
- **Admission control** — Kubernetes admission controllers (Kyverno, OPA Gatekeeper) refuse to schedule pods whose images haven't been scanned or have unmitigated criticals.

```bash
trivy image ghcr.io/myorg/api:1.2.3
grype ghcr.io/myorg/api:1.2.3
```

Both produce a list like:

```
ghcr.io/myorg/api:1.2.3 (debian 12.5)
=====================================
Total: 47 (UNKNOWN: 0, LOW: 12, MEDIUM: 28, HIGH: 6, CRITICAL: 1)

Library          Vulnerability         Severity   Installed   Fixed
libssl3          CVE-2024-XXXX         HIGH       3.0.11-1    3.0.13-1
...
```

Notebook 08 goes deep on container security and how to reduce the surface area (smaller bases, non-root users, distroless, dropping capabilities). Scanning is the *detection* half — supply-chain hygiene is the *prevention* half.

In [ ]:
# A standalone registry demo end to end (requires Docker daemon + outbound access)
!docker run -d -p 5000:5000 --name nb07-registry --restart=always registry:2 2>/dev/null || docker start nb07-registry
!sleep 2
!docker pull alpine:3.20
!docker tag alpine:3.20 localhost:5000/alpine:nb07
!docker push localhost:5000/alpine:nb07
!echo '--- registry catalog ---'
!curl -s http://localhost:5000/v2/_catalog
!echo
!echo '--- tags for the alpine repo ---'
!curl -s http://localhost:5000/v2/alpine/tags/list
!echo
!echo '--- raw manifest (the JSON the registry stores) ---'
!curl -s -H 'Accept: application/vnd.oci.image.manifest.v1+json' http://localhost:5000/v2/alpine/manifests/nb07 | head -25
!echo '--- clean up ---'
!docker rm -f nb07-registry
!docker rmi localhost:5000/alpine:nb07 || true